- Meditations, Seneca, Epictetus

In [0]:
import dlt
import pyspark.sql.functions as F

In [0]:
@dlt.table(
    name = "bronze_philosophers",
    table_properties = {'quality' : 'bronze'},
    comment = "Raw addresses data ingested from the source system"
)
def create_bronze_addresses():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/Volumes/end_to_end_genie_room/default/db_landing/")
            .select(
                "*",
                F.col("_metadata.file_path").alias("input_file_path"),
                F.current_timestamp().alias("ingest_timestamp")
            )
    )

In [0]:
@dlt.table(
    name = "silver_philosophers_clean",
    comment = "Cleaned addresses data",
    table_properties = {'quality' : 'silver'}
)
@dlt.expect_or_fail("valid_id","id IS NOT NULL")
@dlt.expect_or_drop("valid_Content","Content IS NOT NULL")
def create_silver_meditations_clean():
    return (
        spark.readStream.table("LIVE.bronze_philosophers")
            .select(
                "id",
                "Content",
                F.col("ingest_timestamp").cast("date")
            )
    )
           

In [0]:
# preserving full history no deletions
dlt.create_streaming_table(
    name = "silver_philosophers",
    comment = "SCD Type 2 addresses data",
    table_properties = {'quality' : 'silver'}
)

In [0]:
dlt.apply_changes(
    target = "silver_philosophers",
    source = "silver_philosophers_clean",
    keys = ["id"],
    sequence_by = "ingest_timestamp",
    stored_as_scd_type = 2 # duplicate no overwrites
)